# 01 — Configuração do ambiente — TEA

Este notebook configura o ambiente computacional local utilizado
para o processamento e análise dos dados do ABIDE.

**Objetivo:** Configurar o ambiente local (VS Code + ambiente virtual), fixar a semente e criar a estrutura de pastas.

**Entradas:** Nenhuma (as bibliotecas vêm de `requirements.txt`, instalado no ambiente virtual).

**Processamento:** Seeds (`SEED = 42`), criação das pastas, verificação de versões e de GPU.

**Saídas:** Pastas `Data/`, `Figures/`, `Models/`, `Results/`, `Docs/` criadas.

Raiz do projeto, caminhos e semente (importados de `src/config.py`)

In [ ]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Criação das pastas do projeto

In [ ]:
criar_pastas()

for nome, caminho in PATHS.items():
    print(f'OK  [{nome}] -> {caminho}')

Checagem das bibliotecas (instaladas via `requirements.txt`)

In [ ]:
import sys
from importlib import metadata

import sklearn, nilearn, xgboost
print('Python:', sys.version.split()[0])
print('Executável:', sys.executable)
print()
print('scikit-learn:', sklearn.__version__)
print('nilearn:', nilearn.__version__)
print('xgboost:', xgboost.__version__)
print()

pacotes = ['numpy', 'pandas', 'scipy', 'matplotlib', 'seaborn', 'scikit-learn',
           'nilearn', 'nibabel', 'xgboost', 'shap', 'joblib', 'tensorflow']
faltando = []
for p in pacotes:
    try:
        print(f'{p:14s}', metadata.version(p))
    except metadata.PackageNotFoundError:
        print(f'{p:14s} NÃO INSTALADO')
        faltando.append(p)

if faltando:
    print('\nPacotes ausentes:', faltando)
    print('Instale as dependências listadas em requirements.txt (ver README.md) com o ambiente virtual ativo.')

Reprodutibilidade (random, NumPy e, se instalado, TensorFlow/Keras)

In [ ]:
from src.utils import definir_sementes

try:
    import tensorflow  # noqa: F401
    _tf_ok = True
except ImportError:
    _tf_ok = False

definir_sementes(SEED, tensorflow=_tf_ok)
print(f'SEED = {SEED} (random, NumPy' + (', TensorFlow)' if _tf_ok else ')'))

Verificação de GPU (opcional — o projeto roda em CPU)

In [ ]:
import shutil
import subprocess

print('--- nvidia-smi ---')
if shutil.which('nvidia-smi'):
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                        '--format=csv,noheader'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print('nvidia-smi não encontrado (sem driver NVIDIA).')

print('\n--- TensorFlow ---')
if _tf_ok:
    from src.utils import info_gpu_tensorflow
    info_gpu_tensorflow()
else:
    print('TensorFlow não instalado.')

print('\n--- XGBoost ---')
from src.utils import detectar_dispositivo_xgboost
detectar_dispositivo_xgboost()

In [ ]:
print('Ambiente configurado.')
print('Raiz do projeto:', BASE_DIR)